# KNeighbors Classifier: Klasifikace pomocí K nejbližších sousedů

## Co je KNeighbors Classifier?

KNeighbors Classifier (klasifikátor K nejbližších sousedů) je neparametrický, příkladově založený (instance-based) algoritmus, který klasifikuje nové datové body na základě "hlasování" jeho K nejbližších sousedů v prostoru příznaků. Je to jeden z nejjednodušších algoritmů strojového učení, který pracuje na principu podobnosti - nový bod je klasifikován na základě většinového rozhodnutí K nejbližších bodů z trénovací množiny.

### Kdy použít KNeighbors Classifier:
- Když máte malé až středně velké datové sady
- Když data mají nelineární rozhraní (hranice) mezi třídami
- Když chcete jednoduchý a intuitivní model
- Když potřebujete rychlé prototypování
- Když nepotřebujete explicitně trénovat model (lazy learning)

### Výhody:
- Jednoduchý a snadno interpretovatelný algoritmus
- Žádný předpoklad o datech (neparametrický)
- Efektivní pro třídy s komplexními hranicemi
- Přirozeně vícetřídový (multi-class)
- Nepotřebuje explicitní fázi trénování

### Nevýhody:
- Výpočetně náročný při predikci na velkých datových sadách
- Citlivý na irelevantní příznaky a měřítko příznaků
- Vyžaduje hodně paměti (musí ukládat všechny trénovací vzorky)
- "Prokletí dimenzionality" - ve vysokodimenzionálním prostoru přestává být koncept "sousednosti" smysluplný
- Problém s nevyvážeností tříd

Pojďme nyní implementovat KNeighbors Classifier na reálných datech.

In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, plot_confusion_matrix
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap
import time

# Nastavení pro reprodukovatelnost
np.random.seed(42)

# Nastavení vizualizace
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Načtení a Průzkum Dat

Pro tuto demonstraci použijeme dataset Iris, který je klasickým datasetem pro klasifikaci. Obsahuje 150 vzorků tří různých druhů kosatců (setosa, versicolor a virginica) a každý vzorek má čtyři příznaky - délku a šířku kališních lístků a korunních plátků.

In [ ]:
# Načtení dataset Iris
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

# Vytvoření pandas DataFrame pro snadnější manipulaci a vizualizaci
iris_df = pd.DataFrame(data=X, columns=feature_names)
iris_df['species'] = pd.Categorical.from_codes(y, target_names)

# Základní informace o datovém souboru
print(f"Tvar dat: {X.shape}")
print(f"Příznaky: {feature_names}")
print(f"Cílové třídy: {target_names}")
print(f"Distribuce tříd: {np.bincount(y)}")

# Zobrazení prvních pěti řádků dat
print("\nUkázka dat:")
display(iris_df.head())

### Vizualizace dat

Podívejme se na rozložení dat a vztahy mezi příznaky. To nám pomůže lépe pochopit, jak by mohl KNN klasifikátor fungovat na těchto datech.

In [ ]:
# Vytvoření párových grafů pro vizualizaci vztahů mezi příznaky
plt.figure(figsize=(12, 10))
sns.pairplot(iris_df, hue='species', height=2.5)
plt.tight_layout()
plt.show()

# Zobrazení korelační matice
plt.figure(figsize=(10, 8))
corr = iris_df.iloc[:, :-1].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='viridis', vmax=1, vmin=-1, annot=True, fmt='.2f')
plt.title('Korelační matice příznaků Iris datasetu')
plt.tight_layout()
plt.show()

### PCA vizualizace

Použijeme PCA (Principal Component Analysis) pro zobrazení dat ve 2D prostoru, což nám umožní lépe pochopit strukturu dat a jak by KNN mohl fungovat.

In [ ]:
# Aplikace PCA pro redukci dimenzionality na 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(StandardScaler().fit_transform(X))

# Vizualizace dat ve 2D prostoru
plt.figure(figsize=(10, 8))
for i, target_name in enumerate(target_names):
    plt.scatter(X_pca[y == i, 0], X_pca[y == i, 1], label=target_name)
    
plt.xlabel('První hlavní komponenta')
plt.ylabel('Druhá hlavní komponenta')
plt.title('PCA projekce Iris datasetu')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Výpis vysvětleného rozptylu
explained_variance = pca.explained_variance_ratio_
print(f"Vysvětlený rozptyl prvními dvěma komponentami: {sum(explained_variance):.4f}")
print(f"Vysvětlený rozptyl po komponentách: {explained_variance}")

## 2. Příprava dat pro modelování

Nyní rozdělíme data na trénovací a testovací sady a provedeme standardizaci, což je důležité pro KNN, protože používá vzdálenosti mezi body.

In [ ]:
# Rozdělení dat na trénovací a testovací množiny
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Tvar trénovacích dat: {X_train.shape}")
print(f"Tvar testovacích dat: {X_test.shape}")
print(f"Distribuce tříd v trénovacích datech: {np.bincount(y_train)}")
print(f"Distribuce tříd v testovacích datech: {np.bincount(y_test)}")

## 3. Implementace základního KNN klasifikátoru

Začneme s jednoduchou implementací KNN klasifikátoru s výchozími parametry. Pak budeme model postupně vylepšovat.

In [ ]:
# Vytvoření a natrénování základního KNN klasifikátoru
knn_basic = KNeighborsClassifier(n_neighbors=5)  # výchozí hodnota K=5
knn_basic.fit(X_train, y_train)

# Predikce na testovacích datech
y_pred_basic = knn_basic.predict(X_test)

# Vyhodnocení modelu
accuracy_basic = accuracy_score(y_test, y_pred_basic)
print(f"Přesnost základního KNN: {accuracy_basic:.4f}")
print("\nKlasifikační report:")
print(classification_report(y_test, y_pred_basic, target_names=target_names))

### Vizualizace matice záměn

Podívejme se na matici záměn (confusion matrix), abychom viděli, jak si model vede v rozpoznávání jednotlivých tříd.

In [ ]:
# Výpočet matice záměn
cm = confusion_matrix(y_test, y_pred_basic)

# Vizualizace matice záměn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap="Blues", fmt="d", xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predikovaná třída')
plt.ylabel('Skutečná třída')
plt.title('Matice záměn pro základní KNN')
plt.tight_layout()
plt.show()

## 4. Vliv parametru K na výkon modelu

Parametr K (počet sousedů) je klíčový pro KNN. Podívejme se, jak různé hodnoty K ovlivňují přesnost modelu.

In [ ]:
# Testování různých hodnot K
k_values = list(range(1, 31))
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    
    # Skóre na trénovacích datech
    train_score = knn.score(X_train, y_train)
    train_scores.append(train_score)
    
    # Skóre na testovacích datech
    test_score = knn.score(X_test, y_test)
    test_scores.append(test_score)

# Vizualizace výsledků
plt.figure(figsize=(12, 6))
plt.plot(k_values, train_scores, 'o-', label='Trénovací přesnost')
plt.plot(k_values, test_scores, 'o-', label='Testovací přesnost')

# Nalezení optimální hodnoty K
optimal_k = k_values[np.argmax(test_scores)]
plt.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimální K = {optimal_k}')

plt.xlabel('Počet sousedů (K)')
plt.ylabel('Přesnost')
plt.title('Vliv počtu sousedů na přesnost KNN')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Optimální hodnota K na základě testovací přesnosti: {optimal_k}")
print(f"Přesnost s optimálním K: {max(test_scores):.4f}")

## 5. Implementace KNN s pipeline a standardizací

KNN je citlivý na měřítko příznaků, proto je důležité standardizovat data. Použijeme pipeline pro správné zpracování dat.

In [ ]:
# Vytvoření pipeline se standardizací a KNN klasifikátorem
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=optimal_k))
])

# Trénování pipeline
pipeline.fit(X_train, y_train)

# Predikce na testovacích datech
y_pred_pipeline = pipeline.predict(X_test)

# Vyhodnocení modelu
accuracy_pipeline = accuracy_score(y_test, y_pred_pipeline)
print(f"Přesnost KNN s pipeline a standardizací: {accuracy_pipeline:.4f}")
print("\nKlasifikační report:")
print(classification_report(y_test, y_pred_pipeline, target_names=target_names))

## 6. Vliv různých metrik vzdálenosti

KNN používá metriky vzdálenosti pro určení "blízkosti" bodů. Podívejme se, jak různé metriky ovlivňují výkon modelu.

In [ ]:
# Testování různých metrik vzdálenosti
metrics = ['euclidean', 'manhattan', 'chebyshev', 'minkowski']
metric_scores = []

for metric in metrics:
    # Vytvoření pipeline s konkrétní metrikou
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=optimal_k, metric=metric))
    ])
    
    # Křížová validace
    scores = cross_val_score(pipeline, X, y, cv=5)
    metric_scores.append(scores.mean())
    
    print(f"Metrika: {metric}, Průměrná přesnost (CV): {scores.mean():.4f}, Std: {scores.std():.4f}")

# Vizualizace výsledků
plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, metric_scores, color=['#2A9D8F', '#E9C46A', '#F4A261', '#E76F51'])

# Přidání hodnot nad sloupce
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.4f}', ha='center', va='bottom')

plt.xlabel('Metrika vzdálenosti')
plt.ylabel('Průměrná přesnost (5-fold CV)')
plt.title('Vliv metriky vzdálenosti na přesnost KNN')
plt.ylim(0.9, 1.0)  # Upraveno pro lepší viditelnost rozdílů
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## 7. Optimalizace hyperparametrů pomocí GridSearchCV

Použijeme GridSearchCV k nalezení optimální kombinace hyperparametrů pro KNN.

In [ ]:
# Definice parametrů pro GridSearchCV
param_grid = {
    'knn__n_neighbors': list(range(1, 20, 2)),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski'],
    'knn__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# Vytvoření základní pipeline
base_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# Inicializace GridSearchCV
grid_search = GridSearchCV(
    base_pipeline, 
    param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1,  # Použít všechna dostupná jádra CPU
    verbose=1
)

# Spuštění vyhledávání
print("Spouštím GridSearchCV pro nalezení optimálních parametrů...")
start_time = time.time()
grid_search.fit(X, y)
search_time = time.time() - start_time
print(f"Vyhledávání dokončeno za {search_time:.2f} sekund.")

# Výsledky
print(f"\nNejlepší parametry: {grid_search.best_params_}")
print(f"Nejlepší skóre přesnosti: {grid_search.best_score_:.4f}")

# Použití nejlepšího modelu
best_knn = grid_search.best_estimator_
best_y_pred = best_knn.predict(X_test)
best_accuracy = accuracy_score(y_test, best_y_pred)

print(f"\nPřesnost nejlepšího modelu na testovacích datech: {best_accuracy:.4f}")

### Vizualizace top 10 kombinací parametrů

In [ ]:
# Převod výsledků do DataFrame pro snadnější manipulaci
results = pd.DataFrame(grid_search.cv_results_)

# Seřazení výsledků podle průměrného skóre
results = results.sort_values(by='rank_test_score')

# Zobrazení top 10 kombinací
top_10 = results.head(10)
print("Top 10 kombinací parametrů:")

# Vytvoření sloupce s čitelným popisem parametrů pro vizualizaci
top_10['params_summary'] = top_10.apply(
    lambda x: f"K={x['param_knn__n_neighbors']}, {x['param_knn__weights']}, {x['param_knn__metric']}", 
    axis=1
)

# Vizualizace top 10 kombinací
plt.figure(figsize=(14, 8))
sns.barplot(data=top_10, x='params_summary', y='mean_test_score')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Kombinace parametrů')
plt.ylabel('Průměrná přesnost (CV)')
plt.title('Top 10 kombinací parametrů podle přesnosti')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## 8. Vizualizace rozhodovacích hranic

Pro lepší pochopení, jak KNN klasifikátor rozděluje prostor příznaků, vizualizujeme jeho rozhodovací hranice ve 2D prostoru (použijeme PCA pro redukci dimenze).

In [ ]:
# Funkce pro vykreslení rozhodovacích hranic
def plot_decision_boundaries(classifier, X, y, title="Rozhodovací hranice KNN"):
    # Transformace dat do 2D pomocí PCA
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)
    
    # Natrénování klasifikátoru na transformovaných datech
    classifier.fit(X_2d, y)
    
    # Nastavení kroku pro mřížku
    h = 0.02  # menší h znamená jemnější mřížku
    
    # Vytvoření mřížky pro zobrazení hranic
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Predikce pro každý bod mřížky
    Z = classifier.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Vykreslení výsledků
    plt.figure(figsize=(12, 8))
    plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.viridis)
    
    # Vykreslení bodů podle třídy
    for i, color in zip(range(len(np.unique(y))), plt.cm.viridis(np.linspace(0, 1, len(np.unique(y))))):
        idx = np.where(y == i)
        plt.scatter(X_2d[idx, 0], X_2d[idx, 1], c=[color], edgecolor='black', label=f'Třída {i}')
    
    plt.xlabel('První hlavní komponenta')
    plt.ylabel('Druhá hlavní komponenta')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Vykreslení rozhodovacích hranic pro základní KNN a optimalizovaný KNN
# Základní KNN (K=5)
plot_decision_boundaries(KNeighborsClassifier(n_neighbors=5), X, y, "Rozhodovací hranice - základní KNN (K=5)")

# Optimalizovaný KNN (nejlepší parametry z GridSearchCV)
best_params = grid_search.best_params_
best_k = best_params['knn__n_neighbors']
best_weights = best_params['knn__weights']
best_metric = best_params['knn__metric']

plot_decision_boundaries(
    KNeighborsClassifier(n_neighbors=best_k, weights=best_weights, metric=best_metric),
    X, y, 
    f"Rozhodovací hranice - optimalizovaný KNN (K={best_k}, weights={best_weights}, metric={best_metric})"
)

# Výpis vysvětleného rozptylu PCA projekce
print(f"Vysvětlený rozptyl prvními dvěma komponentami: {pca.explained_variance_ratio_.sum():.4f}")

## 9. Vliv vah sousedů na rozhodování

KNN může používat různé váhy pro sousedy. Standardně všichni sousedé mají stejnou váhu ('uniform'), ale můžeme také použít váhy založené na vzdálenosti ('distance').

In [ ]:
# Porovnání vlivu vah sousedů
weights_options = ['uniform', 'distance']
weights_scores = []

for weights in weights_options:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=best_k, weights=weights, metric=best_metric))
    ])
    
    # Křížová validace
    scores = cross_val_score(pipeline, X, y, cv=5)
    weights_scores.append((weights, scores.mean(), scores.std()))
    
    print(f"Váhy: {weights}, Průměrná přesnost (CV): {scores.mean():.4f}, Std: {scores.std():.4f}")
    
    # Vykreslení rozhodovacích hranic pro různé váhy
    plot_decision_boundaries(
        KNeighborsClassifier(n_neighbors=best_k, weights=weights, metric=best_metric),
        X, y, 
        f"Rozhodovací hranice - KNN (K={best_k}, weights={weights}, metric={best_metric})"
    )

## 10. Křivky učení

Podívejme se na křivky učení, abychom pochopili, jak model funguje s různými velikostmi trénovací množiny.

In [ ]:
# Vytvoření optimálního modelu na základě GridSearchCV
optimal_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=best_k, weights=best_weights, metric=best_metric))
])

# Generování křivek učení
train_sizes, train_scores, test_scores = learning_curve(
    optimal_knn, X, y, 
    train_sizes=np.linspace(0.1, 1.0, 10), 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1
)

# Výpočet průměrů a směrodatných odchylek
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

# Vizualizace křivek učení
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='#2A9D8F', label='Trénovací skóre')
plt.plot(train_sizes, test_mean, 'o-', color='#E9C46A', label='Validační skóre')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='#2A9D8F')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='#E9C46A')
plt.xlabel('Počet trénovacích vzorků')
plt.ylabel('Přesnost')
plt.title('Křivky učení pro KNN')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()

## 11. Srovnání KNN s jinými klasifikátory

Porovnejme výkon KNN s jinými populárními klasifikátory na stejných datech.

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Seznam klasifikátorů pro srovnání
classifiers = [
    ('KNN (optimální)', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=best_k, weights=best_weights, metric=best_metric))
    ])),
    ('SVM', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(random_state=42))
    ])),
    ('Rozhodovací strom', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(random_state=42))
    ])),
    ('Random Forest', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(random_state=42))
    ])),
    ('Logistická regrese', Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42, max_iter=1000))
    ]))
]

# Provedení křížové validace pro každý klasifikátor
results = []
for name, classifier in classifiers:
    cv_scores = cross_val_score(classifier, X, y, cv=5)
    results.append((name, cv_scores.mean(), cv_scores.std()))
    print(f"{name}: Přesnost = {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

# Vizualizace výsledků
names = [r[0] for r in results]
means = [r[1] for r in results]
stds = [r[2] for r in results]

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(names)), means, yerr=stds, align='center', 
               color=['#2A9D8F', '#E9C46A', '#F4A261', '#E76F51', '#264653'])
plt.xticks(range(len(names)), names, rotation=45, ha='right')
plt.xlabel('Klasifikátor')
plt.ylabel('Průměrná přesnost (CV)')
plt.title('Srovnání různých klasifikátorů')
plt.ylim(0.8, 1.05)  # Upraveno pro lepší viditelnost
plt.grid(axis='y')

# Přidání hodnot nad sloupce
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 12. Testování na jiných datasetech

Vyzkoušejme náš optimální KNN model na jiných datasetech, abychom lépe pochopili jeho schopnosti.

In [ ]:
# Funkce pro evaluaci modelu na datasetu
def evaluate_on_dataset(X, y, dataset_name):
    # Rozdělení na trénovací a testovací množiny
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Vytvoření optimálního KNN modelu
    optimal_knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=best_k, weights=best_weights, metric=best_metric))
    ])
    
    # Trénování
    optimal_knn.fit(X_train, y_train)
    
    # Predikce a vyhodnocení
    y_pred = optimal_knn.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"Dataset: {dataset_name}")
    print(f"Počet vzorků: {X.shape[0]}, Počet příznaků: {X.shape[1]}")
    print(f"Přesnost optimálního KNN: {accuracy:.4f}")
    print("Klasifikační report:")
    print(classification_report(y_test, y_pred))
    print("\n" + "-"*50)

# 1. Dataset rakoviny prsu
breast_cancer = load_breast_cancer()
evaluate_on_dataset(breast_cancer.data, breast_cancer.target, "Rakovina prsu")

# 2. Dataset vína
wine = load_wine()
evaluate_on_dataset(wine.data, wine.target, "Víno")

## 13. Analýza výpočetní efektivity

KNN je známý tím, že má rychlý čas trénování, ale pomalejší predikce. Podívejme se na to podrobněji.

In [ ]:
from sklearn.datasets import make_classification

# Generování umělých dataset různých velikostí
dataset_sizes = [100, 500, 1000, 5000, 10000]
num_features = 10

# Slovník pro ukládání časů
results = {
    'dataset_size': [],
    'train_time': [],
    'predict_time': []
}

for size in dataset_sizes:
    # Generování dat
    X, y = make_classification(n_samples=size, n_features=num_features, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Vytvoření a trénování KNN
    knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=5))
    ])
    
    # Měření času trénování
    start_time = time.time()
    knn.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Měření času predikce
    start_time = time.time()
    knn.predict(X_test)
    predict_time = time.time() - start_time
    
    # Uložení výsledků
    results['dataset_size'].append(size)
    results['train_time'].append(train_time)
    results['predict_time'].append(predict_time)
    
    print(f"Velikost datasetu: {size}")
    print(f"Čas trénování: {train_time:.4f}s")
    print(f"Čas predikce: {predict_time:.4f}s")
    print("-"*30)

# Vizualizace výsledků
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(results['dataset_size'], results['train_time'], 'o-', color='#2A9D8F')
plt.xlabel('Velikost datasetu')
plt.ylabel('Čas trénování [s]')
plt.title('Škálovatelnost trénování KNN')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(results['dataset_size'], results['predict_time'], 'o-', color='#E9C46A')
plt.xlabel('Velikost datasetu')
plt.ylabel('Čas predikce [s]')
plt.title('Škálovatelnost predikce KNN')
plt.grid(True)

plt.tight_layout()
plt.show()

## 14. Závislostech rychlosti predikce na algoritmu vyhledávání sousedů

KNN implementace v scikit-learn nabízí různé algoritmy pro hledání nejbližších sousedů. Podívejme se, jak ovlivňují výkon.

In [ ]:
# Testování různých algoritmů vyhledávání
algorithms = ['brute', 'kd_tree', 'ball_tree']

# Generování většího datasetu pro lepší měření
X_large, y_large = make_classification(n_samples=5000, n_features=10, random_state=42)
X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(X_large, y_large, test_size=0.2, random_state=42)

# Standardizace dat
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_large)
X_test_scaled = scaler.transform(X_test_large)

for algorithm in algorithms:
    # Vytvoření KNN s konkrétním algoritmem
    knn = KNeighborsClassifier(n_neighbors=5, algorithm=algorithm)
    
    # Měření času trénování
    start_time = time.time()
    knn.fit(X_train_scaled, y_train_large)
    train_time = time.time() - start_time
    
    # Měření času predikce
    start_time = time.time()
    y_pred = knn.predict(X_test_scaled)
    predict_time = time.time() - start_time
    
    # Výpočet přesnosti
    accuracy = accuracy_score(y_test_large, y_pred)
    
    print(f"Algoritmus: {algorithm}")
    print(f"Čas trénování: {train_time:.4f}s")
    print(f"Čas predikce: {predict_time:.4f}s")
    print(f"Přesnost: {accuracy:.4f}")
    print("-"*30)

# Vizualizace výsledků
alg_results = {
    'algorithm': algorithms,
    'train_time': [],
    'predict_time': []
}

for algorithm in algorithms:
    knn = KNeighborsClassifier(n_neighbors=5, algorithm=algorithm)
    
    start_time = time.time()
    knn.fit(X_train_scaled, y_train_large)
    train_time = time.time() - start_time
    
    start_time = time.time()
    knn.predict(X_test_scaled)
    predict_time = time.time() - start_time
    
    alg_results['train_time'].append(train_time)
    alg_results['predict_time'].append(predict_time)

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.bar(alg_results['algorithm'], alg_results['train_time'], color='#2A9D8F')
for i, v in enumerate(alg_results['train_time']):
    plt.text(i, v + 0.0001, f'{v:.4f}s', ha='center')
plt.xlabel('Algoritmus')
plt.ylabel('Čas trénování [s]')
plt.title('Čas trénování podle algoritmu')
plt.grid(axis='y')

plt.subplot(1, 2, 2)
plt.bar(alg_results['algorithm'], alg_results['predict_time'], color='#E9C46A')
for i, v in enumerate(alg_results['predict_time']):
    plt.text(i, v + 0.0001, f'{v:.4f}s', ha='center')
plt.xlabel('Algoritmus')
plt.ylabel('Čas predikce [s]')
plt.title('Čas predikce podle algoritmu')
plt.grid(axis='y')

plt.tight_layout()
plt.show()

## 15. Závěr

### Shrnutí poznatků o KNeighbors Classifier

V tomto notebooku jsme prozkoumali KNeighbors Classifier, který je založen na metodě k-nejbližších sousedů, jednom z nejintuitivnějších algoritmů pro klasifikaci.

**Klíčové poznatky:**

1. **Jednoduchost a efektivita** - KNN je konceptuálně jednoduchý, ale překvapivě efektivní pro mnoho klasifikačních úloh, zejména s optimalizovanými parametry.

2. **Parametr K** - Počet sousedů (K) je klíčovým parametrem, který výrazně ovlivňuje výkon modelu. Příliš malý K může vést k přeučení, příliš velký K k podučení.

3. **Důležitost standardizace** - KNN je citlivý na měřítko příznaků, proto je standardizace dat kritická pro dobré výsledky.

4. **Metriky vzdálenosti** - Různé metriky vzdálenosti (euclidean, manhattan, atd.) mohou poskytovat různé výsledky v závislosti na povaze dat.

5. **Váhy sousedů** - Použití vah založených na vzdálenosti může zlepšit výkon, zejména při vyšších hodnotách K.

6. **Algoritmy hledání sousedů** - Pro velké datasety je volba efektivního algoritmu hledání sousedů ('ball_tree', 'kd_tree') důležitá pro optimalizaci času predikce.

7. **Výpočetní nároky** - KNN má minimální čas trénování, ale čas predikce roste s velikostí trénovacího datasetu, což může být problém pro velké aplikace.

8. **Konkurenceschopný výkon** - S optimálními parametry může KNN dosáhnout výkonu srovnatelného s komplexnějšími modely jako SVM nebo Random Forest.

### Doporučení pro použití KNN v praxi:

1. **Volba K** - Vždy hledejte optimální hodnotu K pomocí křížové validace, většinou se pohybuje mezi 3-10 v závislosti na velikosti datasetu.

2. **Standardizace dat** - Vždy standardizujte příznaky před použitím KNN.

3. **Redukce dimenzionality** - Pro vysokodimenzionální data zvažte předchozí redukci dimenzionality (PCA, t-SNE), která může zlepšit výkon.

4. **Algoritmus vyhledávání** - Pro velké datasety používejte 'kd_tree' nebo 'ball_tree' algoritmy namísto 'brute'.

5. **Vážené hlasování** - Pro složitější hranice mezi třídami zvažte použití weights='distance'.

6. **Omezení velikosti** - Pamatujte, že KNN ukládá celou trénovací množinu, proto pro velmi velké datasety zvažte alternativy nebo podvzorkování.

7. **Rychlé prototypování** - KNN je vynikající pro rychlé prototypování a jako výchozí model před přechodem na složitější techniky.

KNN zůstává cenným nástrojem v sadě nástrojů strojového učení, zejména pro úlohy, kde jsou vztahy mezi daty místně konzistentní a kde je důležitá interpretovatelnost rozhodování.